In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor
import difflib
import warnings
import time
import os
import sys
import platform
from scipy import stats
from scipy.stats import wilcoxon, ttest_rel, shapiro, friedmanchisquare
warnings.filterwarnings('ignore')


# ====================================================================
# GLOBAL CONFIG
# ====================================================================
OPT_SEEDS    = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]   # 10 seeds
EVAL_BUDGET  = 300
PRIMARY_SEED = 42
BOUNDARY_TOL = 0.01

# --- SPEED KNOBS (added) ---
SEARCH_ITER         = 150       # trees during search (was 300)
SEARCH_OD_WAIT      = 30        # early stopping patience during search
FINAL_PROBE_ITER    = 1000      # trees during final refit probe
FINAL_OD_WAIT       = 50
LEARNING_CURVE_ITER = 500
PROGRESS_EVERY      = 25
DEBUG_PRINT_ERRORS  = True      # ← set False once you've confirmed why evals fail

DEFAULT_PARAMS = {
    'learning_rate': 0.03,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'bagging_temperature': 1.0,
    'random_strength': 1.0,
    'border_count': 128,
    'rsm': 1.0,
}


# ====================================================================
# SEARCH SPACE
# ====================================================================
PARAM_BOUNDS = {
    'learning_rate':       [0.01, 0.3],
    'depth':               [3, 10],
    'l2_leaf_reg':         [1, 10],
    'bagging_temperature': [0, 1],
    'random_strength':     [0.5, 5],
    'border_count':        [32, 255],
    'rsm':                 [0.5, 1.0],
}
INT_PARAMS = ['depth', 'border_count']

# Safe default metrics — used so `.format()` never crashes on None
SAFE_METRICS = {'rmse': float('nan'), 'mae': float('nan'),
                'r2': float('-inf'), 'best_iter': 0}


def sample_random_params(rng):
    p = {}
    for k, (lb, ub) in PARAM_BOUNDS.items():
        if k in INT_PARAMS:
            p[k] = int(rng.integers(lb, ub + 1))
        else:
            p[k] = float(rng.uniform(lb, ub))
    return p


# ====================================================================
# BOUNDARY CHECK
# ====================================================================
def boundary_check(best_params, tol=BOUNDARY_TOL):
    rows = []
    for k, v in best_params.items():
        lb, ub = PARAM_BOUNDS[k]
        rng = ub - lb
        d_low, d_high = v - lb, ub - v
        if d_low <= tol * rng:    status = "LOWER BOUND HIT"
        elif d_high <= tol * rng: status = "UPPER BOUND HIT"
        else:                     status = "No"
        rows.append({'Hyperparameter': k, 'Value': v, 'Lower': lb, 'Upper': ub,
                     'Distance to nearest bound': min(d_low, d_high),
                     'Boundary status': status})
    return pd.DataFrame(rows)


def print_boundary_check(best_params, tol=BOUNDARY_TOL):
    df = boundary_check(best_params, tol)
    print("\nBOUNDARY CHECK")
    print("-" * 80)
    for _, r in df.iterrows():
        vs = f"{r['Value']:.4f}" if isinstance(r['Value'], float) else f"{r['Value']}"
        print(f"  {r['Hyperparameter']:22s} {vs:>10s}   → {r['Boundary status']}")
    return df


# ====================================================================
# OBJECTIVE FUNCTION  (verbose on failure)
# ====================================================================
def evaluate_params(params, X_tr, y_tr, X_val, y_val, cat_features, seed,
                    iterations=SEARCH_ITER, early_stopping=SEARCH_OD_WAIT):
    try:
        model = CatBoostRegressor(
            learning_rate=params['learning_rate'],
            depth=int(params['depth']),
            l2_leaf_reg=params['l2_leaf_reg'],
            bagging_temperature=params['bagging_temperature'],
            random_strength=params['random_strength'],
            border_count=int(params['border_count']),
            rsm=params['rsm'],
            iterations=iterations,
            verbose=False,
            thread_count=-1,
            random_seed=seed,
            allow_writing_files=False,
            od_type='Iter',
            od_wait=early_stopping,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
            use_best_model=True,
            cat_features=cat_features if cat_features else []
        )
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        mae  = mean_absolute_error(y_val, preds)
        r2   = r2_score(y_val, preds)
        fitness = rmse + mae + (1 - r2)
        best_iter = model.get_best_iteration() or iterations
        return fitness, {'rmse': rmse, 'mae': mae, 'r2': r2,
                         'best_iter': best_iter}, params
    except Exception as e:
        if DEBUG_PRINT_ERRORS:
            print(f"      [eval FAILED] {type(e).__name__}: {e} | params={params}")
        return float('inf'), {'rmse': float('inf'), 'mae': float('inf'),
                              'r2': -float('inf'),
                              'best_iter': iterations}, params


# ====================================================================
# MOWCA  (with progress + None-safe logging)
# ====================================================================
class MOWCA:
    def __init__(self, n_streams=25, n_rivers=5, n_seas=1,
                 eval_budget=EVAL_BUDGET, dmax=1e-5,
                 evaporation_rate=0.01, raining_rate=0.1, random_seed=42):
        self.n_streams = n_streams
        self.n_rivers  = n_rivers
        self.n_seas    = n_seas
        self.pop_size  = n_seas + n_rivers + n_streams
        self.dmax      = dmax
        self.evaporation_rate = evaporation_rate
        self.raining_rate     = raining_rate
        self.rng  = np.random.default_rng(random_seed)
        self.seed = random_seed
        self.max_iter = max(1, eval_budget // self.pop_size - 2)

    def _initialize_population(self):
        return [sample_random_params(self.rng) for _ in range(self.pop_size)]

    def _flow_intensity(self, fvals):
        fvals = np.asarray(fvals, dtype=float)
        mx, mn = fvals.max(), fvals.min()
        if mx == mn:
            return np.ones_like(fvals) / len(fvals)
        n = np.maximum((mx - fvals) / (mx - mn), 0)
        return n / n.sum()

    def _move(self, src, dst, C):
        out = {}
        for k, (lb, ub) in PARAM_BOUNDS.items():
            if k in INT_PARAMS:
                out[k] = int(np.clip(src[k] + int(C * (dst[k] - src[k])), lb, ub))
            else:
                out[k] = float(np.clip(src[k] + C * (dst[k] - src[k]), lb, ub))
        return out

    def _evaporate_rain(self, pop, sea_idx, river_indices):
        new_pop = [p.copy() for p in pop]
        sea = pop[sea_idx]
        for r_idx in river_indices:
            r = pop[r_idx]
            d = np.sqrt(sum((r[k] - sea[k]) ** 2 for k in PARAM_BOUNDS))
            if d < self.dmax or self.rng.random() < self.evaporation_rate:
                for s_idx in range(len(new_pop)):
                    if s_idx == sea_idx or s_idx in river_indices:
                        continue
                    np_ = {}
                    for k, (lb, ub) in PARAM_BOUNDS.items():
                        if k in INT_PARAMS:
                            v = sea[k] + int(self.rng.integers(-2, 3))
                            np_[k] = int(np.clip(v, lb, ub))
                        else:
                            v = sea[k] + self.rng.normal(0, 0.05 * (ub - lb))
                            np_[k] = float(np.clip(v, lb, ub))
                    new_pop[s_idx] = np_
        return new_pop

    def _log(self, tag, evals, best, t0):
        el = time.time() - t0
        rate = evals / el if el > 0 else 0
        eta = (EVAL_BUDGET - evals) / rate if rate > 0 else 0
        m = best.get('metrics') or SAFE_METRICS
        r2_s   = m.get('r2',   float('-inf'))
        rmse_s = m.get('rmse', float('nan'))
        print(f"    [{tag}] eval {evals:>3}/{EVAL_BUDGET} | "
              f"fit={best.get('fitness', float('inf')):.4f} | "
              f"R²={r2_s:.4f} | RMSE={rmse_s:.4f} | {el:.0f}s | ETA {eta:.0f}s")

    def optimize(self, X_tr, y_tr, X_val, y_val, cat_features, tag="MOWCA"):
        pop = self._initialize_population()
        fvals, mets = [], []
        best = {'fitness': float('inf'), 'params': None,
                'metrics': dict(SAFE_METRICS)}
        evals = 0
        n_failed = 0
        t0 = time.time()

        print(f"    [{tag}] pop={self.pop_size}, gens={self.max_iter}, "
              f"budget={EVAL_BUDGET}, trees/eval={SEARCH_ITER}")

        for ind in pop:
            f, m, _ = evaluate_params(ind, X_tr, y_tr, X_val, y_val,
                                      cat_features, self.seed)
            fvals.append(f); mets.append(m); evals += 1
            if not np.isfinite(f): n_failed += 1
            if f < best['fitness']:
                best = {'fitness': f, 'params': ind.copy(), 'metrics': m}
            if evals % PROGRESS_EVERY == 0:
                self._log(tag, evals, best, t0)

        order = np.argsort(fvals)
        pop   = [pop[i]   for i in order]
        fvals = [fvals[i] for i in order]
        mets  = [mets[i]  for i in order]

        sea_idx = 0
        river_indices  = list(range(1, min(self.n_rivers + 1, self.pop_size)))
        stream_indices = list(range(len(river_indices) + 1, self.pop_size))

        for _ in range(self.max_iter):
            if evals >= EVAL_BUDGET:
                break
            fi = self._flow_intensity(fvals)
            prob_list = np.array([fi[sea_idx]] + [fi[r] for r in river_indices])
            prob_list = prob_list / prob_list.sum()

            new_pop = [pop[sea_idx].copy()]
            for r_idx in river_indices:
                C = self.rng.uniform(0, 2)
                new_pop.append(self._move(pop[r_idx], pop[sea_idx], C))
            for s_idx in stream_indices:
                try:
                    dest = int(self.rng.choice([sea_idx] + river_indices, p=prob_list))
                except ValueError:
                    dest = int(self.rng.choice([sea_idx] + river_indices))
                C = self.rng.uniform(0, 2)
                new_pop.append(self._move(pop[s_idx], pop[dest], C))

            new_f, new_m = [], []
            for ind in new_pop:
                if evals >= EVAL_BUDGET:
                    break
                f, m, _ = evaluate_params(ind, X_tr, y_tr, X_val, y_val,
                                          cat_features, self.seed)
                new_f.append(f); new_m.append(m); evals += 1
                if not np.isfinite(f): n_failed += 1
                if f < best['fitness']:
                    best = {'fitness': f, 'params': ind.copy(), 'metrics': m}
                if evals % PROGRESS_EVERY == 0:
                    self._log(tag, evals, best, t0)

            merged = sorted(zip(fvals + new_f, pop + new_pop, mets + new_m),
                            key=lambda t: t[0])[:self.pop_size]
            fvals = [t[0] for t in merged]
            pop   = [t[1] for t in merged]
            mets  = [t[2] for t in merged]

            pop = self._evaporate_rain(pop, sea_idx, river_indices)
            self.dmax -= self.dmax / max(1, self.max_iter)

            m = best.get('metrics') or SAFE_METRICS
            print(f"    [{tag}] gen update | evals={evals} | "
                  f"fit={best.get('fitness', float('inf')):.4f} | "
                  f"R²={m.get('r2', float('-inf')):.4f} | {time.time()-t0:.0f}s")

        print(f"    [{tag}] DONE | {evals} evals ({n_failed} failed) "
              f"in {time.time()-t0:.1f}s")

        # --- CRITICAL: fallback if every evaluation failed ---
        if best['params'] is None:
            print(f"    [{tag}] WARNING: all evaluations failed. "
                  f"Falling back to DEFAULT_PARAMS.")
            best['params'] = dict(DEFAULT_PARAMS)

        return best, evals


# ====================================================================
# RANDOM SEARCH  (with progress + fallback)
# ====================================================================
def random_search(X_tr, y_tr, X_val, y_val, cat_features, seed,
                  budget=EVAL_BUDGET, tag="RS"):
    rng = np.random.default_rng(seed)
    best = {'fitness': float('inf'), 'params': None,
            'metrics': dict(SAFE_METRICS)}
    evals = 0
    n_failed = 0
    t0 = time.time()
    print(f"    [{tag}] budget={budget}, trees/eval={SEARCH_ITER}")
    for _ in range(budget):
        p = sample_random_params(rng)
        f, m, _ = evaluate_params(p, X_tr, y_tr, X_val, y_val, cat_features, seed)
        evals += 1
        if not np.isfinite(f): n_failed += 1
        if f < best['fitness']:
            best = {'fitness': f, 'params': p.copy(), 'metrics': m}
        if evals % PROGRESS_EVERY == 0:
            el = time.time() - t0
            rate = evals / el if el > 0 else 0
            eta = (budget - evals) / rate if rate > 0 else 0
            m_s = best.get('metrics') or SAFE_METRICS
            print(f"    [{tag}] eval {evals:>3}/{budget} | "
                  f"fit={best.get('fitness', float('inf')):.4f} | "
                  f"R²={m_s.get('r2', float('-inf')):.4f} | {el:.0f}s | ETA {eta:.0f}s")
    print(f"    [{tag}] DONE | {evals} evals ({n_failed} failed) "
          f"in {time.time()-t0:.1f}s")

    if best['params'] is None:
        print(f"    [{tag}] WARNING: all evaluations failed. "
              f"Falling back to DEFAULT_PARAMS.")
        best['params'] = dict(DEFAULT_PARAMS)
    return best, evals


# ====================================================================
# [FIX 1] FINAL TRAINING WITHOUT TEST-SET LEAKAGE  (guarded)
# ====================================================================
def train_final_model_no_leak(params, X_dev, y_dev, X_test, y_test,
                              cat_features, seed, val_fraction=0.2):
    # --- Guard: if params is None, fall back to DEFAULT_PARAMS ---
    if params is None:
        print("      [train_final] params=None → using DEFAULT_PARAMS")
        params = dict(DEFAULT_PARAMS)

    # --- Guard: fill any missing key from DEFAULT_PARAMS ---
    for k, v in DEFAULT_PARAMS.items():
        params.setdefault(k, v)

    X_tr_in, X_va_in, y_tr_in, y_va_in = train_test_split(
        X_dev, y_dev, test_size=val_fraction, random_state=seed)

    probe_params = {
        'learning_rate': params['learning_rate'],
        'depth': int(params['depth']),
        'l2_leaf_reg': params['l2_leaf_reg'],
        'bagging_temperature': params['bagging_temperature'],
        'random_strength': params['random_strength'],
        'border_count': int(params['border_count']),
        'rsm': params['rsm'],
        'iterations': FINAL_PROBE_ITER,
        'thread_count': -1,
        'random_seed': seed,
        'allow_writing_files': False,
        'verbose': False,
        'od_type': 'Iter',
        'od_wait': FINAL_OD_WAIT,
    }
    probe = CatBoostRegressor(**probe_params)
    probe.fit(
        X_tr_in, y_tr_in,
        eval_set=[(X_va_in, y_va_in)],
        verbose=False, use_best_model=True,
        cat_features=cat_features if cat_features else []
    )
    best_iter = probe.get_best_iteration() or FINAL_PROBE_ITER
    best_iter = max(50, int(best_iter))

    final_params = dict(probe_params)
    final_params['iterations'] = best_iter
    final_model = CatBoostRegressor(**final_params)
    final_model.fit(
        X_dev, y_dev,
        verbose=False,
        cat_features=cat_features if cat_features else []
    )

    tr_pred = final_model.predict(X_dev)
    te_pred = final_model.predict(X_test)
    tr = {'R²':   r2_score(y_dev, tr_pred),
          'RMSE': np.sqrt(mean_squared_error(y_dev, tr_pred)),
          'MAE':  mean_absolute_error(y_dev, tr_pred)}
    te = {'R²':   r2_score(y_test, te_pred),
          'RMSE': np.sqrt(mean_squared_error(y_test, te_pred)),
          'MAE':  mean_absolute_error(y_test, te_pred)}
    return final_model, tr, te, best_iter


# ====================================================================
# [FIX 5] LEARNING-CURVE ANALYSIS
# ====================================================================
def learning_curve_analysis(base_params, X, y, cat_features,
                            train_sizes=np.array([0.2, 0.4, 0.6, 0.8, 1.0]),
                            cv=5, seed=42, iterations=LEARNING_CURVE_ITER):
    kf = KFold(n_splits=cv, shuffle=True, random_state=seed)
    n_total = len(X)
    fracs = np.asarray(train_sizes)
    abs_sizes = (fracs * n_total).astype(int)

    rows = []
    for frac, n in zip(fracs, abs_sizes):
        tr_r2, va_r2, va_rmse = [], [], []
        for tr_idx, va_idx in kf.split(X):
            X_tr_f = X.iloc[tr_idx]; y_tr_f = y.iloc[tr_idx]
            X_va_f = X.iloc[va_idx]; y_va_f = y.iloc[va_idx]
            if n < len(X_tr_f):
                sub_rng = np.random.default_rng(seed)
                sub_idx = sub_rng.choice(len(X_tr_f), n, replace=False)
                X_tr_s = X_tr_f.iloc[sub_idx]; y_tr_s = y_tr_f.iloc[sub_idx]
            else:
                X_tr_s, y_tr_s = X_tr_f, y_tr_f

            p = base_params.copy()
            p.update({'iterations': iterations, 'verbose': False,
                      'thread_count': -1, 'random_seed': seed,
                      'allow_writing_files': False,
                      'od_type': 'Iter', 'od_wait': SEARCH_OD_WAIT})
            m = CatBoostRegressor(**p)
            m.fit(X_tr_s, y_tr_s,
                  eval_set=[(X_va_f, y_va_f)],
                  verbose=False, use_best_model=True,
                  cat_features=cat_features if cat_features else [])

            tr_r2.append(r2_score(y_tr_s, m.predict(X_tr_s)))
            va_pred = m.predict(X_va_f)
            va_r2.append(r2_score(y_va_f, va_pred))
            va_rmse.append(np.sqrt(mean_squared_error(y_va_f, va_pred)))

        rows.append({
            'Train fraction': frac,
            'Train size': n,
            'Train R² (mean)': np.mean(tr_r2),
            'Train R² (SD)':   np.std(tr_r2),
            'Val R² (mean)':   np.mean(va_r2),
            'Val R² (SD)':     np.std(va_r2),
            'Val RMSE (mean)': np.mean(va_rmse),
            'Overfit gap':     np.mean(tr_r2) - np.mean(va_r2),
        })
    return pd.DataFrame(rows)


# ====================================================================
# [FIX 4] SOFTWARE VERSIONS
# ====================================================================
def collect_software_versions():
    v = {'Python': sys.version.split()[0], 'Platform': platform.platform(),
         'NumPy': np.__version__, 'Pandas': pd.__version__}
    for pkg, label in [('scipy','SciPy'), ('sklearn','Scikit-learn'),
                       ('catboost','CatBoost'), ('xgboost','XGBoost'),
                       ('matplotlib','Matplotlib'), ('seaborn','Seaborn'),
                       ('shap','SHAP'), ('lime','LIME')]:
        try:
            v[label] = getattr(__import__(pkg), '__version__', 'unknown')
        except Exception:
            v[label] = 'not installed'
    return v


# ====================================================================
# [FIX 3] PERCENTAGE IMPROVEMENT
# ====================================================================
def percentage_improvement(default_metrics, mowca_metrics):
    rows = []
    for metric in ['R²', 'RMSE', 'MAE']:
        d, m = default_metrics[metric], mowca_metrics[metric]
        if metric == 'R²':
            imp = (m - d) / abs(d) * 100 if d != 0 else np.nan
        else:
            imp = (d - m) / d * 100 if d != 0 else np.nan
        rows.append({'Metric': metric, 'Default': d, 'MOWCA': m,
                     'Improvement (%)': imp})
    return pd.DataFrame(rows)


# ====================================================================
# MAIN
# ====================================================================
print("=" * 80)
print("MOWCA vs. RANDOM SEARCH — IDENTICAL BUDGET, 10 SEEDS")
print("=" * 80)

data_path   = r"D:\2026 Work\My Papers\SCM-based concrete\Modelling of the data\Data\Data.csv"
results_dir = r"D:\2026 Work\My Papers\SCM-based concrete\1st Revision\Plots\Models\Catboost + MOWCA"
os.makedirs(results_dir, exist_ok=True)
print(f"\nResults folder: {results_dir}")

# ---------- [FIX 4] Software versions ----------
versions = collect_software_versions()
with open(f"{results_dir}/Software_Versions.txt", "w", encoding="utf-8") as f:
    for k, v in versions.items():
        f.write(f"{k}: {v}\n")
print("✓ Software_Versions.txt saved")

# ---------- Load & clean data ----------
df = pd.read_csv(data_path, encoding='ISO-8859-1')
df.columns = df.columns.str.strip()
print(f"\nData shape: {df.shape}")

target_col = 'Cylinder compressive strength (MPa)'
if target_col not in df.columns:
    for alt in ['Cylinder compressive strength', 'Compressive strength (MPa)',
                'Compressive strength', 'Strength (MPa)', 'Cylinder strength']:
        matches = difflib.get_close_matches(alt, df.columns, n=1, cutoff=0.6)
        if matches:
            target_col = matches[0]; break
print(f"Target column: {target_col}")

X = df.drop(columns=[target_col])
y = df[target_col]

for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].fillna('missing')
    else:
        X[col] = X[col].fillna(X[col].median())

numeric_cols = X.select_dtypes(include=[np.number]).columns
if len(numeric_cols):
    z = np.abs(stats.zscore(X[numeric_cols]))
    keep = (z < 3).all(axis=1)
    X, y = X[keep], y[keep]
    print(f"After outlier removal: {X.shape}")

df_clean = X.copy()
df_clean[target_col] = y.values
print(f"Cleaned data: {df_clean.shape}")

# ---------- [FIX 6] f′c statistics ----------
fc = df_clean[target_col]
fc_stats = {
    'f′c minimum (MPa)': float(fc.min()),
    'f′c maximum (MPa)': float(fc.max()),
    'f′c mean (MPa)':    float(fc.mean()),
    'f′c median (MPa)':  float(fc.median()),
    'f′c SD (MPa)':      float(fc.std(ddof=1)),
    'N samples':         int(len(fc)),
}
with open(f"{results_dir}/Target_Statistics.txt", "w", encoding="utf-8") as f:
    for k, v in fc_stats.items():
        f.write(f"{k}: {v}\n")
pd.DataFrame([{'Variable': 'f′c (MPa)', 'Mean': fc_stats['f′c mean (MPa)'],
               'Minimum': fc_stats['f′c minimum (MPa)'],
               'Maximum': fc_stats['f′c maximum (MPa)'],
               'Median':  fc_stats['f′c median (MPa)'],
               'SD':      fc_stats['f′c SD (MPa)'],
               'N':       fc_stats['N samples']}]
).to_csv(f"{results_dir}/Table_2_Target_Statistics.csv", index=False)
print("✓ Table_2_Target_Statistics.csv saved")

# ---------- Table 3b ----------
bounds_tbl = pd.DataFrame([
    {'Hyperparameter': k, 'Lower Bound': lb, 'Upper Bound': ub,
     'Type': 'Integer' if k in INT_PARAMS else 'Continuous'}
    for k, (lb, ub) in PARAM_BOUNDS.items()
])
bounds_tbl.to_csv(f"{results_dir}/Table_3b_SearchSpace.csv", index=False)
print("✓ Table_3b_SearchSpace.csv saved")

# ====================================================================
# REPEATED RUNS
# ====================================================================
print("\n" + "=" * 80)
print(f"RUNNING {len(OPT_SEEDS)} REPEATS PER OPTIMIZER (budget = {EVAL_BUDGET})")
print("=" * 80)

all_records = []

for seed in OPT_SEEDS:
    print(f"\n{'#'*80}\n# SEED = {seed}\n{'#'*80}")

    X_dev, X_test, y_dev, y_test = train_test_split(
        df_clean.drop(columns=[target_col]), df_clean[target_col],
        test_size=0.2, random_state=seed)
    X_tr_m, X_val, y_tr_m, y_val = train_test_split(
        X_dev, y_dev, test_size=0.3, random_state=seed)
    cat_feats = [i for i, c in enumerate(X_tr_m.columns)
                 if X_tr_m[c].dtype == 'object']

    # ---------- MOWCA ----------
    t0 = time.time()
    mowca = MOWCA(random_seed=seed, eval_budget=EVAL_BUDGET)
    mowca_best, mowca_evals = mowca.optimize(
        X_tr_m, y_tr_m, X_val, y_val, cat_feats, tag=f"MOWCA s{seed}")
    mowca_time = time.time() - t0

    _, mowca_tr, mowca_te, mowca_best_iter = train_final_model_no_leak(
        mowca_best['params'], X_dev, y_dev, X_test, y_test, cat_feats, seed)

    print(f"  MOWCA        | evals={mowca_evals} | best_iter={mowca_best_iter} "
          f"| time={mowca_time:.1f}s | test R²={mowca_te['R²']:.4f} "
          f"RMSE={mowca_te['RMSE']:.4f} MAE={mowca_te['MAE']:.4f}")

    bcheck = boundary_check(mowca_best['params'])
    boundary_flags = "; ".join(
        f"{r['Hyperparameter']}={r['Boundary status']}"
        for _, r in bcheck.iterrows() if r['Boundary status'] != 'No') or "none"

    all_records.append({
        'Seed': seed, 'Optimizer': 'MOWCA',
        'Train R²': mowca_tr['R²'], 'Test R²': mowca_te['R²'],
        'Train RMSE': mowca_tr['RMSE'], 'Test RMSE': mowca_te['RMSE'],
        'Train MAE': mowca_tr['MAE'], 'Test MAE': mowca_te['MAE'],
        'Evals': mowca_evals, 'Time (s)': mowca_time,
        'Best iteration': mowca_best_iter,
        'Boundary hits': boundary_flags,
        'best_lr': mowca_best['params']['learning_rate'],
        'best_depth': int(mowca_best['params']['depth']),
        'best_l2': mowca_best['params']['l2_leaf_reg'],
        'best_bag': mowca_best['params']['bagging_temperature'],
        'best_rstr': mowca_best['params']['random_strength'],
        'best_border': int(mowca_best['params']['border_count']),
        'best_rsm': mowca_best['params']['rsm'],
    })

    # ---------- Random Search ----------
    t0 = time.time()
    rs_best, rs_evals = random_search(X_tr_m, y_tr_m, X_val, y_val, cat_feats,
                                      seed=seed, budget=EVAL_BUDGET,
                                      tag=f"RS    s{seed}")
    rs_time = time.time() - t0

    _, rs_tr, rs_te, rs_best_iter = train_final_model_no_leak(
        rs_best['params'], X_dev, y_dev, X_test, y_test, cat_feats, seed)

    print(f"  RandomSearch | evals={rs_evals} | best_iter={rs_best_iter} "
          f"| time={rs_time:.1f}s | test R²={rs_te['R²']:.4f} "
          f"RMSE={rs_te['RMSE']:.4f} MAE={rs_te['MAE']:.4f}")

    rs_bcheck = boundary_check(rs_best['params'])
    rs_bflags = "; ".join(
        f"{r['Hyperparameter']}={r['Boundary status']}"
        for _, r in rs_bcheck.iterrows() if r['Boundary status'] != 'No') or "none"

    all_records.append({
        'Seed': seed, 'Optimizer': 'RandomSearch',
        'Train R²': rs_tr['R²'], 'Test R²': rs_te['R²'],
        'Train RMSE': rs_tr['RMSE'], 'Test RMSE': rs_te['RMSE'],
        'Train MAE': rs_tr['MAE'], 'Test MAE': rs_te['MAE'],
        'Evals': rs_evals, 'Time (s)': rs_time,
        'Best iteration': rs_best_iter,
        'Boundary hits': rs_bflags,
        'best_lr': rs_best['params']['learning_rate'],
        'best_depth': int(rs_best['params']['depth']),
        'best_l2': rs_best['params']['l2_leaf_reg'],
        'best_bag': rs_best['params']['bagging_temperature'],
        'best_rstr': rs_best['params']['random_strength'],
        'best_border': int(rs_best['params']['border_count']),
        'best_rsm': rs_best['params']['rsm'],
    })

runs_df = pd.DataFrame(all_records)
runs_df.to_csv(f"{results_dir}/Optimizer_Comparison_Runs.csv", index=False)
print(f"\n✓ Optimizer_Comparison_Runs.csv saved")

# ====================================================================
# SUMMARY
# ====================================================================
summary_rows = []
for opt in ['RandomSearch', 'MOWCA']:
    sub = runs_df[runs_df['Optimizer'] == opt]
    summary_rows.append({
        'Optimizer': opt,
        'Test R² (mean ± SD)':   f"{sub['Test R²'].mean():.4f} ± {sub['Test R²'].std():.4f}",
        'Test RMSE (mean ± SD)': f"{sub['Test RMSE'].mean():.4f} ± {sub['Test RMSE'].std():.4f}",
        'Test MAE (mean ± SD)':  f"{sub['Test MAE'].mean():.4f} ± {sub['Test MAE'].std():.4f}",
        'R² min':    f"{sub['Test R²'].min():.4f}",
        'R² max':    f"{sub['Test R²'].max():.4f}",
        'R² spread': f"{sub['Test R²'].max() - sub['Test R²'].min():.4f}",
    })
summary_tbl = pd.DataFrame(summary_rows)
print("\n" + summary_tbl.to_string(index=False))
summary_tbl.to_csv(f"{results_dir}/Optimizer_Comparison_Summary.csv", index=False)
print("✓ Optimizer_Comparison_Summary.csv saved")

# ====================================================================
# BOUNDARY CHECK — AGGREGATED
# ====================================================================
boundary_records = []
for opt in ['MOWCA', 'RandomSearch']:
    sub = runs_df[runs_df['Optimizer'] == opt]
    for param in PARAM_BOUNDS.keys():
        col = {
            'learning_rate': 'best_lr', 'depth': 'best_depth',
            'l2_leaf_reg': 'best_l2', 'bagging_temperature': 'best_bag',
            'random_strength': 'best_rstr', 'border_count': 'best_border',
            'rsm': 'best_rsm',
        }[param]
        if col not in sub.columns: continue
        vals = sub[col].values
        lb, ub = PARAM_BOUNDS[param]
        rng = ub - lb
        n_lo = int(np.sum(np.abs(vals - lb) <= BOUNDARY_TOL * rng))
        n_hi = int(np.sum(np.abs(vals - ub) <= BOUNDARY_TOL * rng))
        boundary_records.append({
            'Optimizer': opt, 'Hyperparameter': param,
            'Lower': lb, 'Upper': ub,
            'Mean value (across seeds)': float(np.mean(vals)),
            'Min value': float(np.min(vals)),
            'Max value': float(np.max(vals)),
            f'Runs at LOWER (tol={BOUNDARY_TOL})': n_lo,
            f'Runs at UPPER (tol={BOUNDARY_TOL})': n_hi,
        })
boundary_df = pd.DataFrame(boundary_records)
print("\n" + boundary_df.to_string(index=False))
boundary_df.to_csv(f"{results_dir}/Boundary_Check_AllSeeds.csv", index=False)

# Primary-seed report
prim_mowca = runs_df[(runs_df['Seed'] == PRIMARY_SEED) &
                     (runs_df['Optimizer'] == 'MOWCA')].iloc[0]
primary_params = {
    'learning_rate': prim_mowca['best_lr'],
    'depth': prim_mowca['best_depth'],
    'l2_leaf_reg': prim_mowca['best_l2'],
    'bagging_temperature': prim_mowca['best_bag'],
    'random_strength': prim_mowca['best_rstr'],
    'border_count': prim_mowca['best_border'],
    'rsm': prim_mowca['best_rsm'],
}
primary_boundary = print_boundary_check(primary_params)
primary_boundary.to_csv(f"{results_dir}/Boundary_Check_PrimarySeed.csv", index=False)

# ====================================================================
# % IMPROVEMENT
# ====================================================================
default_records = []
for seed in OPT_SEEDS:
    X_dev, X_test, y_dev, y_test = train_test_split(
        df_clean.drop(columns=[target_col]), df_clean[target_col],
        test_size=0.2, random_state=seed)
    cat_feats = [i for i, c in enumerate(X_dev.columns)
                 if X_dev[c].dtype == 'object']
    _, _, def_te, _ = train_final_model_no_leak(
        dict(DEFAULT_PARAMS), X_dev, y_dev, X_test, y_test, cat_feats, seed)
    default_records.append({'Seed': seed,
                            'Test R²': def_te['R²'],
                            'Test RMSE': def_te['RMSE'],
                            'Test MAE': def_te['MAE']})

default_df = pd.DataFrame(default_records).sort_values('Seed').reset_index(drop=True)
default_df.to_csv(f"{results_dir}/Default_Baseline_Runs.csv", index=False)

mowca_df = runs_df[runs_df['Optimizer'] == 'MOWCA'].sort_values('Seed').reset_index(drop=True)
mean_default = {'R²':   default_df['Test R²'].mean(),
                'RMSE': default_df['Test RMSE'].mean(),
                'MAE':  default_df['Test MAE'].mean()}
mean_mowca   = {'R²':   mowca_df['Test R²'].mean(),
                'RMSE': mowca_df['Test RMSE'].mean(),
                'MAE':  mowca_df['Test MAE'].mean()}
imp_df = percentage_improvement(mean_default, mean_mowca)
print("\n% improvement:")
print(imp_df.to_string(index=False))
imp_df.to_csv(f"{results_dir}/Percentage_Improvement_Table.csv", index=False)

# ====================================================================
# SIGNIFICANCE TESTS
# ====================================================================
rs_df = runs_df[runs_df['Optimizer'] == 'RandomSearch'].sort_values('Seed').reset_index(drop=True)
m_r2,   r_r2   = mowca_df['Test R²'].values,   rs_df['Test R²'].values
m_rmse, r_rmse = mowca_df['Test RMSE'].values, rs_df['Test RMSE'].values
m_mae,  r_mae  = mowca_df['Test MAE'].values,  rs_df['Test MAE'].values
d_r2           = default_df['Test R²'].values

def fmt_p(p): return "< 1e-4" if p < 1e-4 else f"{p:.4f}"
sig_lines = []

print("\n[Test 1] MOWCA vs Random Search")
paired_tests = []
for name, a, b in [('Test R²', m_r2, r_r2),
                   ('Test RMSE', m_rmse, r_rmse),
                   ('Test MAE', m_mae, r_mae)]:
    w, wp = wilcoxon(a, b, alternative='two-sided')
    t, tp = ttest_rel(a, b)
    line = (f"  {name:10s} | MOWCA={a.mean():.4f} RS={b.mean():.4f} "
            f"Δ={a.mean()-b.mean():+.4f} | t p={fmt_p(tp)} Wilcoxon p={fmt_p(wp)}")
    print(line); sig_lines.append(line)
    paired_tests.append({'Metric': name, 'MOWCA_mean': a.mean(), 'RS_mean': b.mean(),
                         'Delta': a.mean()-b.mean(), 't_p': tp, 'wilcoxon_p': wp})
pd.DataFrame(paired_tests).to_csv(f"{results_dir}/Significance_MOWCA_vs_RS.csv", index=False)

print("\n[Test 2] MOWCA vs Default")
default_tests = []
for name, a, b in [('Test R²', m_r2, d_r2),
                   ('Test RMSE', m_rmse, default_df['Test RMSE'].values),
                   ('Test MAE', m_mae, default_df['Test MAE'].values)]:
    w, wp = wilcoxon(a, b, alternative='two-sided')
    t, tp = ttest_rel(a, b)
    line = (f"  {name:10s} | MOWCA={a.mean():.4f} Def={b.mean():.4f} "
            f"Δ={a.mean()-b.mean():+.4f} | t p={fmt_p(tp)} Wilcoxon p={fmt_p(wp)}")
    print(line); sig_lines.append(line)
    default_tests.append({'Metric': name, 'MOWCA_mean': a.mean(),
                          'Default_mean': b.mean(),
                          'Delta': a.mean()-b.mean(),
                          't_p': tp, 'wilcoxon_p': wp})
pd.DataFrame(default_tests).to_csv(f"{results_dir}/Significance_MOWCA_vs_Default.csv", index=False)

fried_stat, fried_p = friedmanchisquare(-m_r2, -r_r2, -d_r2)
sig_lines.append(f"Friedman χ²={fried_stat:.4f}, p={fmt_p(fried_p)}")
print(sig_lines[-1])

shapiro_lines = []
for name, arr in [('MOWCA', m_r2), ('RS', r_r2), ('Default', d_r2)]:
    w, p = shapiro(arr)
    ln = f"  {name:8s} W={w:.4f}, p={fmt_p(p)}"
    shapiro_lines.append(ln)

with open(f"{results_dir}/Significance_Tests.txt", "w", encoding="utf-8") as f:
    f.write("Significance Tests\n" + "="*80 + "\n\n")
    f.write("\n".join(sig_lines) + "\n")
    f.write("\nShapiro-Wilk:\n" + "\n".join(shapiro_lines) + "\n")

# ====================================================================
# LEARNING CURVE
# ====================================================================
prim = mowca_df[mowca_df['Seed'] == PRIMARY_SEED].iloc[0]
lc_params = {
    'learning_rate': prim['best_lr'], 'depth': int(prim['best_depth']),
    'l2_leaf_reg': prim['best_l2'], 'bagging_temperature': prim['best_bag'],
    'random_strength': prim['best_rstr'],
    'border_count': int(prim['best_border']), 'rsm': prim['best_rsm'],
}
X_lc = df_clean.drop(columns=[target_col])
y_lc = df_clean[target_col]
cat_lc = [i for i, c in enumerate(X_lc.columns) if X_lc[c].dtype == 'object']

lc_df = learning_curve_analysis(lc_params, X_lc, y_lc, cat_lc,
                                train_sizes=np.array([0.2, 0.4, 0.6, 0.8, 1.0]),
                                cv=5, seed=PRIMARY_SEED)
print(lc_df.to_string(index=False))
lc_df.to_csv(f"{results_dir}/Learning_Curve.csv", index=False)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig_lc, ax_lc = plt.subplots(figsize=(9, 6))
ax_lc.plot(lc_df['Train fraction']*100, lc_df['Train R² (mean)'], 'o-',
           color='tab:blue', label='Training R²')
ax_lc.fill_between(lc_df['Train fraction']*100,
                   lc_df['Train R² (mean)'] - lc_df['Train R² (SD)'],
                   lc_df['Train R² (mean)'] + lc_df['Train R² (SD)'],
                   alpha=0.2, color='tab:blue')
ax_lc.plot(lc_df['Train fraction']*100, lc_df['Val R² (mean)'], 's-',
           color='tab:red', label='Validation R²')
ax_lc.fill_between(lc_df['Train fraction']*100,
                   lc_df['Val R² (mean)'] - lc_df['Val R² (SD)'],
                   lc_df['Val R² (mean)'] + lc_df['Val R² (SD)'],
                   alpha=0.2, color='tab:red')
ax_lc.set_xlabel('Training-set fraction (%)', fontweight='bold')
ax_lc.set_ylabel('R² score', fontweight='bold')
ax_lc.set_title('Learning Curve — MOWCA-CatBoost', fontweight='bold')
ax_lc.legend(); ax_lc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{results_dir}/8_Learning_Curve.png", dpi=500, bbox_inches='tight')
plt.close()
print("✓ 8_Learning_Curve.png saved")

# ---- Boxplots ----
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, m in zip(axes, ['Test R²', 'Test RMSE', 'Test MAE']):
    data = [runs_df[runs_df['Optimizer'] == o][m].values
            for o in ['RandomSearch', 'MOWCA']]
    bp = ax.boxplot(data, labels=['RandomSearch', 'MOWCA'],
                    patch_artist=True, widths=0.5)
    for patch, c in zip(bp['boxes'], ['tab:gray', 'tab:blue']):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    ax.set_title(m, fontweight='bold'); ax.grid(True, alpha=0.3, axis='y')
plt.suptitle(f'Optimizer comparison (n={len(OPT_SEEDS)} seeds, budget={EVAL_BUDGET})',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f"{results_dir}/9_Optimizer_Comparison_Boxplots.png", dpi=500, bbox_inches='tight')
plt.close()
print("✓ 9_Optimizer_Comparison_Boxplots.png saved")

# ====================================================================
# CONSOLIDATED SUMMARY
# ====================================================================
with open(f"{results_dir}/Full_Summary.txt", "w", encoding="utf-8") as f:
    f.write("MOWCA-CatBoost — Full Summary\n" + "="*80 + "\n\n")
    f.write("Software versions:\n")
    for k, v in versions.items(): f.write(f"  {k}: {v}\n")
    f.write("\nf′c statistics:\n")
    for k, v in fc_stats.items(): f.write(f"  {k}: {v}\n")
    f.write("\nSearch space:\n" + bounds_tbl.to_string(index=False) + "\n")
    f.write("\nSummary:\n" + summary_tbl.to_string(index=False) + "\n")
    f.write("\n% improvement:\n" + imp_df.to_string(index=False) + "\n")
    f.write("\nBoundary check:\n" + boundary_df.to_string(index=False) + "\n")
    f.write("\nLearning curve:\n" + lc_df.to_string(index=False) + "\n")
    f.write("\nSignificance:\n" + "\n".join(sig_lines) + "\n")
    f.write("\nShapiro:\n" + "\n".join(shapiro_lines) + "\n")
print("✓ Full_Summary.txt saved")

print("\n" + "="*80)
print("ALL DONE. FILES SAVED TO:")
print(results_dir)
print("="*80)

MOWCA vs. RANDOM SEARCH — IDENTICAL BUDGET, 10 SEEDS

Results folder: D:\2026 Work\My Papers\SCM-based concrete\1st Revision\Plots\Models\Catboost + MOWCA
✓ Software_Versions.txt saved

Data shape: (1456, 9)
Target column: Cylinder compressive strength (MPa)
After outlier removal: (1347, 8)
Cleaned data: (1347, 9)
✓ Table_2_Target_Statistics.csv saved
✓ Table_3b_SearchSpace.csv saved

RUNNING 10 REPEATS PER OPTIMIZER (budget = 300)

################################################################################
# SEED = 42
################################################################################
    [MOWCA s42] pop=31, gens=7, budget=300, trees/eval=150
    [MOWCA s42] eval  25/300 | fit=10.8563 | R²=0.8530 | RMSE=6.1639 | 18s | ETA 199s
    [MOWCA s42] eval  50/300 | fit=10.8563 | R²=0.8530 | RMSE=6.1639 | 32s | ETA 158s
    [MOWCA s42] gen update | evals=62 | fit=10.8563 | R²=0.8530 | 39s
    [MOWCA s42] eval  75/300 | fit=10.8563 | R²=0.8530 | RMSE=6.1639 | 50s | ETA 149s
  